# 02 - Regression: Opportunity Amount

Benchmark notebook for the `Opportunity Amount USD` target.

Model families compared include linear / robust regressors, binned variants, and XGBoost regression.

Data source: `../../data/intermediate/df_train_stratified.parquet`.


In [1]:
# ISSUE: https://github.com/scikit-learn/scikit-learn/issues/32247
# ISSUE: https://github.com/guillermo-navas-palencia/optbinning/issues/296
import numpy as np
import sklearn.utils.multiclass as multiclass

# Save the original function just in case
_original_type_of_target = multiclass.type_of_target

def patched_type_of_target(y):
    # If it's float32 or float64
    if y.dtype in [np.float32, np.float64]:
        return 'continuous'
    return _original_type_of_target(y)

# Apply the patch
multiclass.type_of_target = patched_type_of_target

In [2]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, PowerTransformer
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression, ElasticNet, HuberRegressor
from sklearn.utils.multiclass import type_of_target

from xgboost import XGBRegressor

from binning import NamedBinningProcess, DataFrameScaler
from statsmodels_api import StatsModelsRegressor

import statsmodels.api as sm
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)

/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_parquet('../../data/intermediate/df_train_stratified.parquet')
print('shape:', df.shape)

shape: (53073, 37)


In [4]:
# Editable feature list (all enabled by default)
FEATURES = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Elapsed Days In Sales Stage',
    'Sales Stage Change Count',
    'Total Days Identified Through Closing',
    'Total Days Identified Through Qualified',
    #'Opportunity Amount USD',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
    'Competitor Type',
    'Ratio Days Identified To Total Days',
    'Ratio Days Validated To Total Days',
    'Ratio Days Qualified To Total Days',
    #'Deal Size Category (USD)',
    #'total_days_zero',
    #'opportunity_amount_weirdness',
    #'flag_0_days',
    #'flag_ratio_problem',
    #'flag_zero_opportunity_amount',
    #'flag_outlier_opportunity_amount',
    #'flag_outlier_total_days',
    #'flag_totally_repeated_row',
    #'flag_partially_repeated_row',
    #'partial_repeat_is_latest_id_appearance',
    #'flag_only_identified',
    #'flag_weirdness_over_75th_pct',
    #'problem_count',
]

TARGET = 'Opportunity Amount USD'

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

X = df[FEATURES].copy()
y = df[TARGET].astype(np.float32).copy()
print('n_features:', len(FEATURES))
print('target mean (Opportunity Amount USD):', y.mean())

n_features: 15
target mean (Opportunity Amount USD): 94148.13


In [5]:
type_of_target(y)

'continuous'

In [6]:
#X["flag_0_days"] = (X["Total Days Identified Through Closing"] == 0).astype(str)
#X["flag_ratio_problem"] = (X["Total Days Identified Through Closing"] == 0).astype(str)


In [7]:
X_cols = list(X.columns)
X_categorical_cols = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Competitor Type',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
#    'flag_0_days',
#    'flag_ratio_problem',
]
X_numerical_cols = [c for c in X_cols if c not in X_categorical_cols]

In [8]:
def get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols):
    return ColumnTransformer([
        ('numerical', Pipeline([
            ("imputer", SimpleImputer(strategy=imputer_strategy_numerical)),
            ("scaler", scaler)
        ]), numerical_cols),
        ('categorical', Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
            ("ohe", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]).set_output(transform="pandas")
    
def get_transformed_regressor_log1p(model):
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )
    
def get_power_transformed_regressor(model):
    return TransformedTargetRegressor(
        regressor=model,
        transformer=PowerTransformer(method='yeo-johnson', standardize=True)
    )

In [9]:
experiment_grid = {
    "classic_linear_standard_scaler": Pipeline([
        ("preprocessing",get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsRegressor(sm.OLS))
    ]),
    "classic_linear_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsRegressor(sm.OLS))
    ]),
    "huber_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', HuberRegressor())
    ]),
    "elasticnet_standard_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', ElasticNet(l1_ratio=0.5, max_iter=10000))
    ]),
    "linear_binned_ohe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="bin_ohe"
        )),
        ('model', StatsModelsRegressor(sm.OLS))
    ]),
    "linear_binned_woe_standard_scaler": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="mean"
        )),
        ('scaler', DataFrameScaler(StandardScaler())),
        ('model', StatsModelsRegressor(sm.OLS))
    ]),
    "linear_binned_woe_robust_scaler": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="mean"
        )),
        ('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsRegressor(sm.OLS))
    ]),
    "linear_binned_woe_robust_scaler": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="mean"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsRegressor(sm.OLS))
    ]),
    "elastic_binned_woe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="mean"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', LinearRegression())
    ]),
    "xgboost": Pipeline([
        ('preprocessing', get_classic_preprocessor(scaler=None, imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', XGBRegressor(eval_metric='logloss'))
    ])
}

In [10]:
cv = 5
skf = KFold(n_splits=cv, shuffle=True, random_state=42)

results = []

In [11]:
outer_bar = tqdm(experiment_grid.items(), total=len(experiment_grid), desc="Experiments", position=0)

for exp_name, pipeline in outer_bar:
    outer_bar.set_postfix_str(exp_name)

    inner_bar = tqdm(
        enumerate(skf.split(X, y), start=1),
        total=cv,
        desc=f"{exp_name} folds",
        leave=False,
        position=1
    )

    for fold, (train_idx, valid_idx) in inner_bar:
        X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
        X_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]

        y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
        y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

        model = clone(pipeline)

        try:
            model.fit(X_train, y_train)

            y_pred = model.predict(X_valid)

            results.append({
                "experiment": exp_name,
                "fold": fold,
                "r2": r2_score(y_valid, y_pred),
                "mae": mean_absolute_error(y_valid, y_pred),
                "mse": mean_squared_error(y_valid, y_pred),
                "mape": mean_absolute_percentage_error(y_valid, y_pred),
                "status": "ok",
                "error": None,
            })

        except Exception as e:
            results.append({
                "experiment": exp_name,
                "fold": fold,
                "r2": np.nan,
                "mae": np.nan,
                "mse": np.nan,
                "mape": np.nan,
                "status": "failed",
                "error": str(e),
            })

Experiments:   0%|          | 0/9 [00:00<?, ?it/s]

Experiments:   0%|          | 0/9 [00:00<?, ?it/s, classic_linear_standard_scaler]

classic_linear_standard_scaler folds:   0%|          | 0/5 [00:00<?, ?it/s]

classic_linear_standard_scaler folds:  20%|██        | 1/5 [00:01<00:04,  1.00s/it]

classic_linear_standard_scaler folds:  40%|████      | 2/5 [00:01<00:02,  1.18it/s]

...

xgboost folds:  60%|██████    | 3/5 [00:04<00:03,  1.61s/it]

xgboost folds:  80%|████████  | 4/5 [00:06<00:01,  1.58s/it]

xgboost folds: 100%|██████████| 5/5 [00:08<00:00,  1.70s/it]

Experiments: 100%|██████████| 9/9 [01:34<00:00,  9.66s/it, xgboost]

Experiments: 100%|██████████| 9/9 [01:34<00:00, 10.46s/it, xgboost]

In [12]:
results_df = pd.DataFrame(results)

summary_df = (
    results_df
    .groupby("experiment", dropna=False)
    .agg(
        r2_mean=("r2", "mean"),
        r2_std=("r2", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        mse_mean=("mse", "mean"),
        mse_std=("mse", "std"),
        mape_mean=("mape", "mean"),
        mape_std=("mape", "std"),
        n_failed=("status", lambda s: (s == "failed").sum()),
    )
    .sort_values("r2_mean", ascending=False)
    .reset_index()
)

results_df, summary_df

(                           experiment  fold        r2           mae  \
 0      classic_linear_standard_scaler     1  0.130638  75308.865127   
 1      classic_linear_standard_scaler     2  0.125975  74543.724303   
 2      classic_linear_standard_scaler     3  0.140580  75192.370615   
 3      classic_linear_standard_scaler     4  0.137896  73129.869143   
 4      classic_linear_standard_scaler     5  0.136439  73138.778179   
 5        classic_linear_robust_scaler     1  0.130638  75308.865127   
 6        classic_linear_robust_scaler     2  0.125975  74543.724303   
 7        classic_linear_robust_scaler     3  0.140580  75192.370615   
 8        classic_linear_robust_scaler     4  0.137896  73129.869143   
 9        classic_linear_robust_scaler     5  0.136439  73138.778179   
 10                huber_robust_scaler     1  0.058932  68676.900458   
 11                huber_robust_scaler     2  0.060983  67698.083861   
 12                huber_robust_scaler     3  0.063898  68796.74

In [13]:
summary_df

,experiment,r2_mean,r2_std,mae_mean,mae_std,mse_mean,mse_std,mape_mean,mape_std,n_failed
0,classic_linear_standard_scaler,0.134305,0.005909,74262.721473,1070.540704,1.557560e+10,8.707191e+08,297.091131,54.859637,0
1,classic_linear_robust_scaler,0.134305,0.005909,74262.721473,1070.540704,1.557560e+10,8.707191e+08,297.091131,54.859637,0
2,linear_binned_ohe,0.134271,0.007220,74254.814519,989.503366,1.557539e+10,8.543479e+08,300.437127,55.170291,0
3,elastic_binned_woe,0.130785,0.008194,74287.262178,1112.589231,1.563849e+10,8.688657e+08,298.657714,58.660308,0
4,linear_binned_woe_standard_scaler,0.130785,0.008194,74287.262178,1112.589231,1.563849e+10,8.688657e+08,298.657714,58.660308,0
5,linear_binned_woe_robust_scaler,0.130785,0.008194,74287.262178,1112.589231,1.563849e+10,8.688657e+08,298.657714,58.660308,0
6,xgboost,0.129360,0.013067,73733.087500,880.282389,1.565922e+10,7.734293e+08,273.477032,40.822789,0
7,elasticnet_standard_scaler,0.066263,0.003013,79640.061998,1240.511108,1.680157e+10,9.746000e+08,261.280429,57.885111,0
8,huber_robust_scaler,0.065812,0.006468,67336.787762,1505.620114,1.681207e+10,1.029491e+09,198.744995,38.035123,0


In [14]:
# SLIDE_DECK_EXPORT_REGRESSION
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold, cross_val_predict

slide_data_dir = Path('../../slidedeck/data')
slide_data_dir.mkdir(parents=True, exist_ok=True)
output_path = slide_data_dir / 'regression_model_report.xlsx'

best_experiment = summary_df.sort_values('r2_mean', ascending=False).iloc[0]['experiment']
best_pipeline = clone(experiment_grid[best_experiment])
best_pipeline.fit(X, y)

cv = skf if 'skf' in globals() else KFold(n_splits=5, shuffle=True, random_state=42)
pred_amount = cross_val_predict(clone(experiment_grid[best_experiment]), X, y, cv=cv)

sample_n = min(len(X), 5000)
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X), size=sample_n, replace=False) if len(X) > sample_n else np.arange(len(X))
X_sample = X.iloc[sample_idx] if hasattr(X, 'iloc') else X[sample_idx]
y_sample = y.iloc[sample_idx] if hasattr(y, 'iloc') else y[sample_idx]
perm = permutation_importance(
    best_pipeline,
    X_sample,
    y_sample,
    n_repeats=5,
    random_state=42,
    scoring='r2',
    n_jobs=1,
)
feature_names = list(X.columns) if hasattr(X, 'columns') else [f'feature_{i}' for i in range(len(perm.importances_mean))]
feature_importance_df = (
    pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    })
    .sort_values('importance_mean', ascending=False)
    .reset_index(drop=True)
)

prediction_export = pd.DataFrame({
    'row_id': list(X.index) if hasattr(X, 'index') else np.arange(len(pred_amount)),
    'actual_amount': y.to_numpy() if hasattr(y, 'to_numpy') else np.asarray(y),
    'predicted_amount': pred_amount,
})
prediction_export['absolute_error'] = (prediction_export['actual_amount'] - prediction_export['predicted_amount']).abs()
prediction_export['ape'] = prediction_export['absolute_error'] / prediction_export['actual_amount'].replace(0, np.nan)

forecast_df = prediction_export.copy()
forecast_df['predicted_amount_clipped'] = forecast_df['predicted_amount'].clip(lower=0)
forecast_summary_df = pd.DataFrame([
    {'metric': 'actual_total_amount', 'value': float(forecast_df['actual_amount'].sum())},
    {'metric': 'predicted_total_amount', 'value': float(forecast_df['predicted_amount_clipped'].sum())},
    {'metric': 'mean_absolute_error', 'value': float(prediction_export['absolute_error'].mean())},
    {'metric': 'median_absolute_percentage_error', 'value': float(prediction_export['ape'].dropna().median())},
])

metadata_df = pd.DataFrame([
    {
        'target': 'Opportunity Amount USD',
        'best_experiment': best_experiment,
        'best_r2_mean': summary_df.iloc[0]['r2_mean'],
        'best_mae_mean': summary_df.iloc[0]['mae_mean'],
        'best_mape_mean': summary_df.iloc[0]['mape_mean'],
        'notes': 'Snapshot regression model used for sizing and forecasting use case.',
    }
])

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='cv_results', index=False)
    summary_df.to_excel(writer, sheet_name='cv_summary', index=False)
    feature_importance_df.to_excel(writer, sheet_name='feature_importance', index=False)
    prediction_export.to_excel(writer, sheet_name='oof_predictions', index=False)
    forecast_summary_df.to_excel(writer, sheet_name='forecast_summary', index=False)
    metadata_df.to_excel(writer, sheet_name='metadata', index=False)

print('saved slide deck regression report:', output_path)
print('best experiment:', best_experiment)


saved slide deck regression report: ../../slidedeck/data/regression_model_report.xlsx
best experiment: classic_linear_standard_scaler
